<a href="https://colab.research.google.com/github/sdazzleberry/MLforQSTOpen_Project_Winter_2025/blob/my_assignments/assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASSIGNMENT 1

In [ ]:
!pip install qiskit qiskit-aer qiskit-experiments numpy scipy pandas plotly tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 629.3/629.3 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.5/378.5 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 13.7 MB/s eta 0:00:00


I used Google Colab with Qiskit Aer simulators to ensure reproducibility and avoid hardware noise. All experiments were run with fixed random seeds.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import depolarizing_error, amplitude_damping_error
from qiskit.quantum_info import DensityMatrix, Statevector

np.random.seed(42)
simulator = AerSimulator(seed_simulator=42)


**Born rule recap and derivation:**

For a quantum state (density matrix)
p
and measurement operators
(Mk)

p(k)=Tr(Mk​ρ)
For this to define valid probabilities, we must have:
1. p(k)>=0
2. ∑k​ p(k) = 1 (total probability sums to 1)

**A) for projective measurements:**


Mk​=Pk​ , (Pk)^2​=Pk​ , Pk†​=Pk ​, k∑​Pk​=I

they normalise as:

k∑​p(k)=k∑​Tr(Pk​ρ)=Tr(k∑​Pk​ρ)=Tr(Iρ)=Tr(ρ)=1

For any projector Pk:

Tr(Pkp)>= 0


because both Pk ​and
ρ are positive semidefinite.


**B) For POVMs:**

Mk = Ek, Ek>= 0, k∑​Ek​=I

and the normalisation happens do to the same trace argument as in projective measurements.

In [ ]:
#numerical completeness check for POVM
import numpy as np

def check_completeness(operators, tol=1e-12):
    dim = operators[0].shape[0]
    I = np.eye(dim)
    residual = sum(operators) - I
    return np.linalg.norm(residual) < tol, np.linalg.norm(residual)


In [ ]:
#numerical completeness check for projectives (example: pauli z)
import numpy as np

# Projectors
P0 = np.array([[1, 0],
               [0, 0]])

P1 = np.array([[0, 0],
               [0, 1]])

# Identity
I = np.eye(2)

# Sum of operators
residual = P0 + P1 - I

# Tolerance
tol = 1e-12

# Boolean completeness check
is_complete = np.linalg.norm(residual) < tol

print("||Σ P_k − I|| =", np.linalg.norm(residual))
print("Projective measurement complete?", is_complete)


||Σ P_k − I|| = 0.0
Projective measurement complete? True


**PROS AND CONS:**

**A) Projective (Pauli) Measurements:**

Pros

*   Simple


*   Experimentally natural


*   Robust to noise


*   Easy inversion


Cons

* Not informationally complete in a single setting

* Requires multiple bases

* Redundant sampling



**B) SIC POVMs**:

Pros

* Informationally complete in one measurement

* Minimal number of outcomes(dsq)

* Symmetric inversion

* Optimal mean-squared error (known results)

Cons

*  To implement physically

* Non-orthogonal outcomes

* More sensitive to miscalibration







In this work, a hybrid measurement model is adopted. Pauli projective measurements are used for data generation and state reconstruction due to their hardware-native implementation and direct support in Qiskit’s tomography framework. In parallel, Symmetric Informationally Complete POVMs (SIC POVMs) are analyzed theoretically as an alternative measurement strategy with minimal outcome cardinality and improved symmetry properties.

This hybrid approach enables reproducible tomography using standard tools while still evaluating the scalability and efficiency benefits of SIC POVMs in future multi-qubit settings.

**Preparation of states:**

All states were prepared using unitary gates acting on the computational ground state
∣
0
⟩
∣0⟩. The corresponding quantum circuits are summarized below.

In [ ]:
def prepare_state(label: str) -> QuantumCircuit:
    qc = QuantumCircuit(1)
    if label == "0":
        pass
    elif label == "1":
        qc.x(0)
    elif label == "+":
        qc.h(0)
    elif label == "-":
        qc.x(0)
        qc.h(0)
    elif label == "i":
        qc.h(0)
        qc.s(0)
    return qc


In [ ]:
import json
import pathlib

STATE_PREPARATION_DATA = {
    "0": {
        "description": "|0> computational basis state",
        "gates": []
    },
    "1": {
        "description": "|1> computational basis state",
        "gates": [
            {"gate": "X", "target": 0}
        ]
    },
    "+": {
        "description": "|+> = (|0> + |1>)/sqrt(2)",
        "gates": [
            {"gate": "H", "target": 0}
        ]
    },
    "-": {
        "description": "|-> = (|0> - |1>)/sqrt(2)",
        "gates": [
            {"gate": "X", "target": 0},
            {"gate": "H", "target": 0}
        ]
    },
    "i": {
        "description": "|i> = (|0> + i|1>)/sqrt(2)",
        "gates": [
            {"gate": "H", "target": 0},
            {"gate": "S", "target": 0}
        ]
    }
}


output_path = pathlib.Path("single_qubit_state_preparation.json")

with open(output_path, "w") as f:
    json.dump(STATE_PREPARATION_DATA, f, indent=2)

print(f"State preparation metadata saved to {output_path.resolve()}")



State preparation metadata saved to /content/single_qubit_state_preparation.json


Single qubit states:
1. Computational basis:
The qubit is initialized in
∣0⟩, no gates are applied.
The Pauli-X gate flips
∣0⟩ to
∣1⟩.

2. Hadamard basis: The Hadamard gate creates an equal superposition of
∣0⟩ and
∣1⟩.

3. Phase offset state:
the phase gate
𝑆
applies a relative phase
𝑖
 to the
∣1⟩ component.

**Mixed state preparation**

In [ ]:
def prepare_depolarized_state(state_label: str, p: float) -> DensityMatrix:
    qc = prepare_state(state_label)

    # Create depolarizing noise
    dep_error = depolarizing_error(p, 1)

    # Apply noise to density matrix
    rho = DensityMatrix(Statevector.from_instruction(qc))
    rho_mixed = rho.evolve(dep_error.to_quantumchannel())

    return rho_mixed


In [ ]:
def prepare_amplitude_damped_state(state_label: str, gamma: float) -> DensityMatrix:
    qc = prepare_state(state_label)

    # Create amplitude damping noise
    amp_error = amplitude_damping_error(gamma)

    # Apply noise
    rho = DensityMatrix(Statevector.from_instruction(qc))
    rho_mixed = rho.evolve(amp_error.to_quantumchannel())

    return rho_mixed


In [ ]:
from typing import Dict, Any
import pathlib
import numpy as np

def build_measurement_model(config_path: pathlib.Path) -> Dict[str, Any]:
    """
    Construct a hybrid single-qubit measurement model.

    - Pauli projective measurements are used operationally for tomography.
    - SIC POVM operators are included for theoretical comparison.

    Returns:
        Dictionary containing operators, completeness checks, and metadata.
    """

    # Pauli matrices
    I = np.eye(2)
    X = np.array([[0, 1], [1, 0]])
    Y = np.array([[0, -1j], [1j, 0]])
    Z = np.array([[1, 0], [0, -1]])


    pauli_projectors = {
        "Z": [
            np.array([[1, 0], [0, 0]]),  # |0><0|
            np.array([[0, 0], [0, 1]])   # |1><1|
        ],
        "X": [
            0.5 * (I + X),               # |+><+|
            0.5 * (I - X)                # |-><-|
        ],
        "Y": [
            0.5 * (I + Y),               # |i><i|
            0.5 * (I - Y)                # |-i><-i|
        ]
    }

    # Completeness check for Pauli measurements
    pauli_completeness = {
        basis: np.allclose(sum(projectors), I)
        for basis, projectors in pauli_projectors.items()
    }

    # SIC POVM
    bloch_vectors = [
        np.array([1, 1, 1]),
        np.array([1, -1, -1]),
        np.array([-1, 1, -1]),
        np.array([-1, -1, 1]),
    ]

    sic_povm = []
    for v in bloch_vectors:
        v = v / np.linalg.norm(v)
        rho = 0.5 * (I + v[0]*X + v[1]*Y + v[2]*Z)
        sic_povm.append(0.25 * rho)

    sic_completeness = np.allclose(sum(sic_povm), I)


    measurement_model = {
        "model_type": "hybrid",
        "pauli_projective": {
            "operators": pauli_projectors,
            "completeness_check": pauli_completeness,
        },
        "sic_povm": {
            "operators": sic_povm,
            "completeness_check": sic_completeness,
        },
        "metadata": {
            "num_qubits": 1,
            "description": (
                "Hybrid measurement model: Pauli projective measurements "
                "used operationally for tomography, with SIC POVM operators "
                "included for theoretical efficiency comparison."
            )
        }
    }

    return measurement_model


In [ ]:
import numpy as np
import plotly.graph_objects as go
from fractions import Fraction

_CUBE_FACES = (
    (0, 1, 2), (0, 2, 3),  # bottom
    (4, 5, 6), (4, 6, 7),  # top
    (0, 1, 5), (0, 5, 4),
    (1, 2, 6), (1, 6, 5),
    (2, 3, 7), (2, 7, 6),
    (3, 0, 4), (3, 4, 7)
 )

def _phase_to_pi_string(angle_rad: float) -> str:
    """Format a phase angle as a simplified multiple of π."""
    if np.isclose(angle_rad, 0.0):
        return "0"
    multiple = angle_rad / np.pi
    frac = Fraction(multiple).limit_denominator(16)
    numerator = frac.numerator
    denominator = frac.denominator
    sign = "-" if numerator < 0 else ""
    numerator = abs(numerator)
    if denominator == 1:
        magnitude = f"{numerator}" if numerator != 1 else ""
    else:
        magnitude = f"{numerator}/{denominator}"
    return f"{sign}{magnitude}π" if magnitude else f"{sign}π"

def plot_density_matrix_histogram(rho, basis_labels=None, title="Density matrix (|ρ_ij| as bar height, phase as color)"):
    """Render a density matrix as a grid of solid histogram bars with phase coloring."""
    rho = np.asarray(rho)
    if rho.ndim != 2 or rho.shape[0] != rho.shape[1]:
        raise ValueError("rho must be a square matrix")

    dim = rho.shape[0]
    mags = np.abs(rho)
    phases = np.angle(rho)
    x_vals = np.arange(dim)
    y_vals = np.arange(dim)

    if basis_labels is None:
        basis_labels = [str(i) for i in range(dim)]

    meshes = []
    colorbar_added = False
    for i in range(dim):
        for j in range(dim):
            height = mags[i, j]
            phase = phases[i, j]
            x0, x1 = i - 0.45, i + 0.45
            y0, y1 = j - 0.45, j + 0.45
            vertices = (
                (x0, y0, 0.0), (x1, y0, 0.0), (x1, y1, 0.0), (x0, y1, 0.0),
                (x0, y0, height), (x1, y0, height), (x1, y1, height), (x0, y1, height)
            )
            x_coords, y_coords, z_coords = zip(*vertices)
            i_idx, j_idx, k_idx = zip(*_CUBE_FACES)
            phase_pi = _phase_to_pi_string(phase)
            mesh = go.Mesh3d(
                x=x_coords,
                y=y_coords,
                z=z_coords,
                i=i_idx,
                j=j_idx,
                k=k_idx,
                intensity=[phase] * len(vertices),
                colorscale="HSV",
                cmin=-np.pi,
                cmax=np.pi,
                showscale=not colorbar_added,
                colorbar=dict(
                    title="phase ",
                    tickvals=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
                    ticktext=["-π", "-π/2", "0", "π/2", "π"]
                ) if not colorbar_added else None,
                opacity=1.0,
                flatshading=False,
                hovertemplate=
                    f"i={i}, j={j}<br>|ρ_ij|={height:.3f}<br>arg(ρ_ij)={phase_pi}<extra></extra>",
                lighting=dict(ambient=0.6, diffuse=0.7)
            )
            meshes.append(mesh)
            colorbar_added = True

    fig = go.Figure(data=meshes)
    fig.update_layout(
        scene=dict(
            xaxis=dict(
                title="i",
                tickmode="array",
                tickvals=x_vals,
                ticktext=basis_labels
            ),
            yaxis=dict(
                title="j",
                tickmode="array",
                tickvals=y_vals,
                ticktext=basis_labels
            ),
            zaxis=dict(title="|ρ_ij|"),
            aspectratio=dict(x=1, y=1, z=0.7)
        ),
        title=title,
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

In [ ]:
from qiskit.quantum_info import DensityMatrix, Statevector

# Visualize ideal reference states
for label in ["0", "1", "+", "-", "i"]:
    qc = prepare_state(label)
    rho = DensityMatrix(Statevector.from_instruction(qc))

    plot_density_matrix_histogram(
        rho.data,
        basis_labels=["0", "1"],
        title=f"Ideal state |{label}⟩ (density matrix)"
    )


In [ ]:
rho_dep = prepare_depolarized_state("+", p=0.3)
rho_amp = prepare_amplitude_damped_state("+", gamma=0.4)

plot_density_matrix_histogram(
    rho_dep.data,
    basis_labels=["0", "1"],
    title="Depolarized |+⟩ state (p = 0.3)"
)

plot_density_matrix_histogram(
    rho_amp.data,
    basis_labels=["0", "1"],
    title="Amplitude damped |+⟩ state (γ = 0.4)"
)


The above code visualises the ideal states and the mixed states prepared as density matrices

**Dataset generation**:

**Random Circuit Generation**

To extend the dataset beyond structured reference states, randomly parameterized single-qubit circuits were generated. Each circuit consists of a sequence of randomly chosen single-qubit rotation gates \(R_x(\theta)\), \(R_y(\theta)\), and \(R_z(\theta)\), with rotation angles sampled uniformly from \([0, 2\pi)\). A fixed random seed ensures reproducibility.

These rotation gates form a universal generating set for single-qubit unitaries, allowing the circuit to prepare generic quantum states across the Bloch sphere. Increasing circuit depth increases state expressiveness but also amplifies sensitivity to phase and coherence, making random circuits a useful stress test for tomographic reconstruction methods.


In [ ]:
import numpy as np
from pathlib import Path
from qiskit_aer import AerSimulator

SEED = 42
SHOTS = 2000

np.random.seed(SEED)
simulator = AerSimulator(seed_simulator=SEED)

DATA_DIR = Path("data/single_qubit")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR



PosixPath('data/single_qubit')

In [ ]:
def generate_random_single_qubit_circuit(depth, seed):
    rng = np.random.default_rng(seed)
    qc = QuantumCircuit(1)

    for _ in range(depth):
        gate = rng.choice(["rx", "ry", "rz"])
        theta = rng.uniform(0, 2 * np.pi)
        if gate == "rx":
            qc.rx(theta, 0)
        elif gate == "ry":
            qc.ry(theta, 0)
        elif gate == "rz":
            qc.rz(theta, 0)

    return qc


def apply_depolarizing(qc, p=0.3):
    rho = DensityMatrix(Statevector.from_instruction(qc))
    channel = depolarizing_error(p, 1).to_quantumchannel()
    return rho.evolve(channel).data

def apply_amplitude_damping(qc, gamma=0.3):
    rho = DensityMatrix(Statevector.from_instruction(qc))
    channel = amplitude_damping_error(gamma).to_quantumchannel()
    return rho.evolve(channel).data


In [ ]:
def measure_in_basis(qc: QuantumCircuit, basis: str) -> QuantumCircuit:
    meas = QuantumCircuit(1)
    meas.compose(qc, inplace=True)

    if basis == "X":
        meas.h(0)
    elif basis == "Y":
        meas.sdg(0)
        meas.h(0)
    elif basis == "Z":
        pass
    else:
        raise ValueError("Basis must be X, Y, or Z")

    meas.measure_all()
    return meas
def counts_to_probabilities(counts):
    return {k: v / SHOTS for k, v in counts.items()}


In [ ]:
def generate_pauli_counts(qc: QuantumCircuit, shots: int = SHOTS):
    results = {}
    for basis in ["X", "Y", "Z"]:
        meas = measure_in_basis(qc, basis)
        job = simulator.run(meas, shots=shots)
        results[basis] = job.result().get_counts()
    return results


In [ ]:
DATASET = {}

reference_states = ["0", "1", "+", "-", "i"]

# Ideal reference states
for label in reference_states:
    qc = prepare_state(label)
    counts = generate_pauli_counts(qc)

    DATASET[f"ideal_{label}"] = {
        "type": "ideal",
        "state": label,
        "shots": SHOTS,
        "seed": SEED,
        "counts": counts,
        "probabilities": {
            b: counts_to_probabilities(c)
            for b, c in counts.items()
        }
    }

# Noisy variants (|+⟩)
qc_plus = prepare_state("+")

DATASET["noisy_depolarizing_plus"] = {
    "type": "noisy",
    "model": "depolarizing",
    "parameter": 0.3,
    "seed": SEED,
    "density_matrix": apply_depolarizing(qc_plus).tolist()
}

DATASET["noisy_amplitude_damping_plus"] = {
    "type": "noisy",
    "model": "amplitude_damping",
    "parameter": 0.3,
    "seed": SEED,
    "density_matrix": apply_amplitude_damping(qc_plus).tolist()
}

# Random circuit
random_qc = generate_random_single_qubit_circuit(depth=6, seed=123)
random_counts = generate_pauli_counts(random_qc)

DATASET["random_single_qubit"] = {
    "type": "random",
    "depth": 6,
    "seed": 123,
    "shots": SHOTS,
    "counts": random_counts,
    "probabilities": {
        b: counts_to_probabilities(c)
        for b, c in random_counts.items()
    }
}




In [ ]:
dataset_path = DATA_DIR / "measurement_dataset.npy"
np.save(dataset_path, DATASET, allow_pickle=True)

dataset_path


PosixPath('data/single_qubit/measurement_dataset.npy')

In [ ]:
loaded_dataset = np.load(dataset_path, allow_pickle=True).item()

loaded_dataset.keys()


dict_keys(['ideal_0', 'ideal_1', 'ideal_+', 'ideal_-', 'ideal_i', 'noisy_depolarizing_plus', 'noisy_amplitude_damping_plus', 'random_single_qubit'])

In [ ]:
dataset = np.load(
    "data/single_qubit/measurement_dataset.npy",
    allow_pickle=True
).item()



In [ ]:
state_label = "+"
pauli_data = dataset[f"ideal_{state_label}"]["counts"]


All measurement datasets, metadata, and reconstructed density matrices were serialized using NumPy `.npy` files. This format natively supports complex-valued arrays and nested Python structures, avoiding lossy conversions and ensuring reproducibility of quantum state data.


In [ ]:
import numpy as np
from qiskit.quantum_info import DensityMatrix, Statevector, state_fidelity

#PAULI MATRICES
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

#Expectation value from counts
def pauli_expectation(counts, shots):
    p0 = counts.get("0", 0) / shots
    p1 = counts.get("1", 0) / shots
    return p0 - p1

#Linear inversion tomography
def linear_inversion_tomography(pauli_counts, shots):
    ex = pauli_expectation(pauli_counts["X"], shots)
    ey = pauli_expectation(pauli_counts["Y"], shots)
    ez = pauli_expectation(pauli_counts["Z"], shots)

    rho = 0.5 * (I + ex * X + ey * Y + ez * Z)
    return rho

#MLE MATRIX
def mle_density_matrix(rho):
    rho = 0.5 * (rho + rho.conj().T)  # Hermitian
    eigvals, eigvecs = np.linalg.eigh(rho)
    eigvals = np.maximum(eigvals, 0)  # enforce positivity
    eigvals /= np.sum(eigvals)        # normalize trace
    return eigvecs @ np.diag(eigvals) @ eigvecs.conj().T

#compare the 2


# Ideal density matrix
rho_ideal = DensityMatrix(
    Statevector.from_instruction(prepare_state(state_label))
).data

# Linear inversion
rho_linear = linear_inversion_tomography(pauli_data, SHOTS)

# MLE
rho_mle = mle_density_matrix(rho_linear)

# Fidelities
fid_linear = state_fidelity(
    rho_linear,
    rho_ideal,
    validate=False
)

fid_mle = state_fidelity(
    rho_mle,
    rho_ideal
)


# Eigenvalues (physicality check)
eig_linear = np.linalg.eigvalsh(rho_linear)
eig_mle = np.linalg.eigvalsh(rho_mle)

print("LINEAR INVERSION")
print("Density matrix:\n", rho_linear)
print("Eigenvalues:", eig_linear)
print("Fidelity:", fid_linear)

print("\n MLE RECONSTRUCTION")
print("Density matrix:\n", rho_mle)
print("Eigenvalues:", eig_mle)
print("Fidelity:", fid_mle)


LINEAR INVERSION
Density matrix:
 [[0.506+0.j    0.5  -0.006j]
 [0.5  +0.006j 0.494+0.j   ]]
Eigenvalues: [-7.19948167e-05  1.00007199e+00]
Fidelity: 1.0000000105153797

 MLE RECONSTRUCTION
Density matrix:
 [[0.50599914+0.j         0.49992802-0.00599914j]
 [0.49992802+0.00599914j 0.49400086+0.j        ]]
Eigenvalues: [-5.55111512e-17  1.00000000e+00]
Fidelity: 0.999928015548268


In [ ]:
# Additional metrics

def trace_distance(rho_a, rho_b):
    """Trace distance = 1/2 || rho_a - rho_b ||_1"""
    diff = rho_a - rho_b
    return 0.5 * np.linalg.norm(diff, ord="nuc")

def bloch_vector(rho):
    """Return Bloch vector (x, y, z) for a single-qubit density matrix"""
    x = np.real(np.trace(rho @ X))
    y = np.real(np.trace(rho @ Y))
    z = np.real(np.trace(rho @ Z))
    return np.array([x, y, z])

# Compute metrics
trace_lin = trace_distance(rho_linear, rho_ideal)
trace_mle = trace_distance(rho_mle, rho_ideal)

bloch_ideal = bloch_vector(rho_ideal)
bloch_lin = bloch_vector(rho_linear)
bloch_mle = bloch_vector(rho_mle)

bloch_err_lin = np.linalg.norm(bloch_lin - bloch_ideal)
bloch_err_mle = np.linalg.norm(bloch_mle - bloch_ideal)

trace_lin, trace_mle, bloch_err_lin, bloch_err_mle


(np.float64(0.008485281374238578),
 np.float64(0.008484365134265731),
 np.float64(0.016970562748477157),
 np.float64(0.01696873026853146))

In [ ]:
import pandas as pd

metrics_table = pd.DataFrame({
    "Method": ["Linear Inversion", "MLE"],
    "Fidelity": [fid_linear, fid_mle],
    "Trace Distance": [trace_lin, trace_mle],
    "Bloch Vector Error": [bloch_err_lin, bloch_err_mle],
})

metrics_table


,Method,Fidelity,Trace Distance,Bloch Vector Error
0,Linear Inversion,1.000000,0.008485,0.016971
1,MLE,0.999928,0.008484,0.016969


In [ ]:
# Save reconstructed density matrices
np.save(DATA_DIR / f"{state_label}_rho_linear.npy", rho_linear)
np.save(DATA_DIR / f"{state_label}_rho_mle.npy", rho_mle)
np.save(DATA_DIR / f"{state_label}_rho_ideal.npy", rho_ideal)

# Save metrics
np.save(DATA_DIR / f"{state_label}_metrics.npy", {
    "fidelity_linear": fid_linear,
    "fidelity_mle": fid_mle,
    "trace_distance_linear": trace_lin,
    "trace_distance_mle": trace_mle,
    "bloch_error_linear": bloch_err_lin,
    "bloch_error_mle": bloch_err_mle,
})


In [ ]:
plot_density_matrix_histogram(
    rho_ideal,
    basis_labels=["0", "1"],
    title="Ideal |+⟩ state"
)

plot_density_matrix_histogram(
    rho_linear,
    basis_labels=["0", "1"],
    title="Linear inversion reconstruction"
)

plot_density_matrix_histogram(
    rho_mle,
    basis_labels=["0", "1"],
    title="MLE reconstruction"
)


For ideal reference states measured with high shot counts and complete Pauli measurements, maximum-likelihood estimation converges to the exact target state. This behavior is expected, as MLE removes only unphysical components of the linear estimate and does not introduce additional noise. Deviations between reconstructed and ideal states become apparent when measurement noise, decoherence, or finite sampling effects are introduced.


Below is the entire tomography for a random circuit:

In [ ]:
# Random circuit parameters
RANDOM_DEPTH = 6
RANDOM_SEED = 123

qc_rand = generate_random_single_qubit_circuit(
    depth=RANDOM_DEPTH,
    seed=RANDOM_SEED
)

qc_rand
pauli_data_rand = generate_pauli_counts(qc_rand)

rho_rand_ideal = DensityMatrix(
    Statevector.from_instruction(qc_rand)
).data



In [ ]:
rho_rand_linear = linear_inversion_tomography(
    pauli_data_rand,
    SHOTS
)
rho_rand_mle = mle_density_matrix(rho_rand_linear)
fid_rand_linear = state_fidelity(
    rho_rand_linear,
    rho_rand_ideal,
    validate=False
)

fid_rand_mle = state_fidelity(
    rho_rand_mle,
    rho_rand_ideal
)

fid_rand_linear, fid_rand_mle


(1.0072471339043483, 0.9999705807829486)

In [ ]:
# Trace distance
trace_rand_linear = 0.5 * np.linalg.norm(
    rho_rand_linear - rho_rand_ideal,
    ord="nuc"
)

trace_rand_mle = 0.5 * np.linalg.norm(
    rho_rand_mle - rho_rand_ideal,
    ord="nuc"
)

# Bloch vectors
bloch_ideal = bloch_vector(rho_rand_ideal)
bloch_linear = bloch_vector(rho_rand_linear)
bloch_mle = bloch_vector(rho_rand_mle)

bloch_err_linear = np.linalg.norm(bloch_linear - bloch_ideal)
bloch_err_mle = np.linalg.norm(bloch_mle - bloch_ideal)

trace_rand_linear, trace_rand_mle, bloch_err_linear, bloch_err_mle


(np.float64(0.009099206777335526),
 np.float64(0.005423948474391548),
 np.float64(0.018198413554671014),
 np.float64(0.010847896948783093))

In [ ]:
import pandas as pd

rand_metrics = pd.DataFrame({
    "Method": ["Linear Inversion", "MLE"],
    "Fidelity": [fid_rand_linear, fid_rand_mle],
    "Trace Distance": [trace_rand_linear, trace_rand_mle],
    "Bloch Vector Error": [bloch_err_linear, bloch_err_mle],
})

rand_metrics


,Method,Fidelity,Trace Distance,Bloch Vector Error
0,Linear Inversion,1.007247,0.009099,0.018198
1,MLE,0.999971,0.005424,0.010848


In [ ]:
np.save(DATA_DIR / "random_rho_ideal.npy", rho_rand_ideal)
np.save(DATA_DIR / "random_rho_linear.npy", rho_rand_linear)
np.save(DATA_DIR / "random_rho_mle.npy", rho_rand_mle)

np.save(DATA_DIR / "random_metrics.npy", {
    "fidelity_linear": fid_rand_linear,
    "fidelity_mle": fid_rand_mle,
    "trace_linear": trace_rand_linear,
    "trace_mle": trace_rand_mle,
    "bloch_error_linear": bloch_err_linear,
    "bloch_error_mle": bloch_err_mle,
})


In [ ]:
plot_density_matrix_histogram(
    rho_rand_ideal,
    basis_labels=["0", "1"],
    title="Ideal state: random single-qubit circuit"
)

plot_density_matrix_histogram(
    rho_rand_linear,
    basis_labels=["0", "1"],
    title="Linear inversion: random circuit"
)

plot_density_matrix_histogram(
    rho_rand_mle,
    basis_labels=["0", "1"],
    title="MLE reconstruction: random circuit"
)




Tomographic reconstruction was performed on a randomly synthesized single-qubit circuit. Unlike Pauli-aligned reference states, the random circuit produces a generic quantum state with distributed phase and coherence. Linear inversion exhibits sensitivity to finite sampling, occasionally yielding non-physical estimates. Maximum-likelihood estimation enforces positivity and unit trace, resulting in improved fidelity, reduced trace distance, and lower Bloch vector error relative to the ideal state.


### Conclusion

In this assignment, we built a complete pipeline for single-qubit quantum state tomography. We prepared several reference states (|0⟩, |1⟩, |+⟩, |−⟩, and a phase-offset state), along with noisy states and randomly generated circuits. Each state was measured using Pauli measurements along the X, Y, and Z axes, and the resulting measurement counts, probabilities, and random seeds were saved for reproducibility.

Quantum states were reconstructed using two methods: linear inversion and maximum-likelihood estimation (MLE). Linear inversion is simple and fast, but it can produce unphysical results, such as density matrices with negative eigenvalues. MLE fixes this by enforcing the physical constraints of a valid quantum state, which leads to more reliable reconstructions. For ideal states measured with many shots, MLE recovered the target state almost exactly. For noisy states and random circuits, MLE produced reconstructions that were close to, but clearly different from, the ideal states.

Reconstruction quality was evaluated using fidelity, trace distance, and Bloch vector error. In addition, density matrix histogram plots were used to visually compare ideal and reconstructed states. Overall, the results show how measurement noise, finite sampling, and circuit complexity affect tomography accuracy, and why MLE is important for obtaining physically meaningful quantum state estimates.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Reflection on various tools and Future Work

 Qiskit was effective for circuit construction and simulation, while NumPy enabled flexible manipulation of density matrices and metrics.

One limitation of this assignment is the focus on single-qubit systems, extending it to multi-qubit tomography would significantly increase computational and experimental complexity. Another open question is how different measurement models, such as SIC POVMs, compare to Pauli measurements in terms of efficiency and robustness under noise. Future improvements could include automated maximum-likelihood optimization, adaptive measurement strategies, and experiments with real hardware backends.


In [ ]:
!zip -r single_qubit_data.zip /content/data/single_qubit


  adding: content/data/single_qubit/ (stored 0%)
  adding: content/data/single_qubit/random_rho_ideal.npy (deflated 34%)
  adding: content/data/single_qubit/+_metrics.npy (deflated 36%)
  adding: content/data/single_qubit/random_rho_linear.npy (deflated 42%)
  adding: content/data/single_qubit/+_rho_mle.npy (deflated 42%)
  adding: content/data/single_qubit/+_rho_linear.npy (deflated 46%)
  adding: content/data/single_qubit/random_rho_mle.npy (deflated 42%)
  adding: content/data/single_qubit/measurement_dataset.npy (deflated 68%)
  adding: content/data/single_qubit/+_rho_ideal.npy (deflated 57%)
  adding: content/data/single_qubit/random_metrics.npy (deflated 34%)
